<a href="https://colab.research.google.com/github/kywch/BubbleView_jsPsych/blob/master/todo_math_monster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install pyathena
!pip install pyathena > /dev/null

# upgrade sqlite3
# https://stackoverflow.com/a/59429952/6729010 
!gdown --id 1BSHIKQ7rFw5BpTq5nw1UZfjPK_7Mpnbi
!mv _sqlite3.cpython-37m-x86_64-linux-gnu.so /usr/lib/python3.7/lib-dynload/
import os
os._exit(00)

ERROR: requests 2.23.0 has requirement urllib3!=1.25.0,!=1.25.1,<1.26,>=1.21.1, but you'll have urllib3 1.26.4 which is incompatible.
ERROR: datascience 0.10.6 has requirement folium==0.2.1, but you'll have folium 0.8.3 which is incompatible.
Downloading...
From: https://drive.google.com/uc?id=1BSHIKQ7rFw5BpTq5nw1UZfjPK_7Mpnbi
To: /content/_sqlite3.cpython-37m-x86_64-linux-gnu.so
6.23MB [00:00, 54.8MB/s]


In [1]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# download the file
import json
key_file = drive.CreateFile({'id': '1cA-QjgQmMN8DmmvWiExlHhzDJZlVPmRa'})
aws_access = json.loads(key_file.GetContentString())

# create athena connection
from pyathena import connect
athena = connect(
    aws_access_key_id = aws_access['aws_access_key_id'],
    aws_secret_access_key = aws_access['aws_secret_access_key'],
    s3_staging_dir = 's3://enuma-dw/',
    region_name = 'us-west-2')

/usr/local/lib/python3.7/dist-packages/requests/__init__.py:91: RequestsDependencyWarning: urllib3 (1.26.4) or chardet (3.0.4) doesn't match a supported version!
  RequestsDependencyWarning)


# Import necessary libraries

In [2]:
import pandas as pd
import numpy as np
import sqlite3

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="white", color_codes=True)
%matplotlib inline

import gc
from datetime import datetime, timedelta

import json

from google.colab import data_table, files

In [3]:
# start look at one-day's data
# think about the whole dataset later
yesterday = datetime.strftime(datetime.now() - timedelta(1), '%Y-%m-%d')
weekago = datetime.strftime(datetime.now() - timedelta(7), '%Y-%m-%d')

extract_start = '2021-01-23'

print(yesterday, ',', weekago, ',', extract_start)

2021-04-28 , 2021-04-22 , 2021-01-23


In [4]:
# mount the google drive
from google.colab import data_table, files, drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# 몬스터 퀴즈 데이터 가져오기 
* 각 몬스터 통과율
* 통과 평균 정답 수
* 실패 시 재도전을 하는가?

version 5.21.8 부터 이런 정보를 직접 볼 수 있음 


In [6]:
query = """
  SELECT 
    MIN(regdate)
    , MIN(version)
  FROM dw.tb_fact_todomath_log
  WHERE server_yyyymmdd > '2021-03'
  AND v_gamename = 'MonsterQuiz'
  AND event = 'tmGameComplete'
  AND v_mode = '4'
  AND v_monsterid IS NOT NULL

"""

%time pd.read_sql(query, athena)


CPU times: user 285 ms, sys: 36.7 ms, total: 321 ms
Wall time: 18 s


,_col0,_col1
0,2021-03-29 04:38:36,5.21.4


In [22]:
query = f"""
  SELECT 
    installid
    , accountid
    , userid
    , platform
    , version
    , regdate
    , event
    , v_gamename
    , v_elapsedtime
    , v_totalproblem
    , v_currentproblem
    , v_missed
    , v_solved
    , v_monsterid
    , v_dailyunitindex
  FROM dw.tb_fact_todomath_log
  WHERE server_yyyymmdd > '2021-03'
    AND v_gamename = 'MonsterQuiz'
    AND event = 'tmGameComplete'
    AND v_mode = '4'
    AND v_monsterid IS NOT NULL
    -- AND todomath_regdate > DATE '2021-04-07'
  ORDER BY 2, 3, 6
"""

%time quiz_df = pd.read_sql(query, athena)
quiz_df

CPU times: user 12.5 s, sys: 541 ms, total: 13 s
Wall time: 54.7 s


,installid,accountid,userid,platform,version,regdate,event,v_gamename,v_elapsedtime,v_totalproblem,v_currentproblem,v_missed,v_solved,v_monsterid,v_dailyunitindex
0,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-17 10:57:27,tmGameComplete,MonsterQuiz,16.877234,5,0,5,0,T.M.0,1-0
1,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-17 10:59:20,tmGameComplete,MonsterQuiz,107.633120,5,5,0,5,T.M.0,1-0
2,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-18 08:11:30,tmGameComplete,MonsterQuiz,27.847017,5,5,2,3,A.M.1,1-1
3,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-21 09:23:16,tmGameComplete,MonsterQuiz,29.845467,5,5,2,3,A.M.2,1-2
4,154F6A41-34C4-40E2-A79E-F6B969152B02,0002a3f0-8bfa-11eb-b441-dd5e1e4be4f4,BCFEEB7C-D70E-4CCE-8DE8-0C5CAA6EAA5B,iOS,5.21.8,2021-04-17 06:01:18,tmGameComplete,MonsterQuiz,44.659782,5,5,0,5,B.M.1,2-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125107,a38f3672-d241-41c1-bc16-9fc4a2b22778,None,ffb34dbf-04a9-488a-b54f-fc5626f33539,android,cn.5.21.8,2021-04-24 13:00:16,tmGameComplete,MonsterQuiz,67.649570,5,2,5,0,B.M.1,2-1
125108,b613fdeb-0a42-4719-90ce-6b1500ac3a66,None,ffe0e48f-f1ae-4416-9a42-f6854219a8bd,android,5.21.8,2021-04-19 20:19:43,tmGameComplete,MonsterQuiz,66.917030,5,5,0,5,T.M.0,2-0
125109,7705d58e-9b8f-45fc-ba40-15edb7c6c2f2,None,ffe86cab-522c-4ca0-ad6b-325426f0c486,android,cn.5.21.8,2021-04-27 11:18:22,tmGameComplete,MonsterQuiz,31.929977,5,5,0,5,T.M.0,2-0
125110,8F239119-060F-4F9E-BA49-CB47812EB022,None,newuser,iOS,5.21.8,2021-04-21 14:29:06,tmGameComplete,MonsterQuiz,93.687320,0,0,0,0,D.M.1,0-0


In [10]:
quiz_df.to_csv('monster_quiz.csv', index=False)

In [15]:
quiz_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33193 entries, 0 to 33192
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   installid         33193 non-null  object        
 1   accountid         33193 non-null  object        
 2   userid            33193 non-null  object        
 3   platform          33193 non-null  object        
 4   version           33193 non-null  object        
 5   regdate           33193 non-null  datetime64[ns]
 6   event             33193 non-null  object        
 7   v_gamename        33193 non-null  object        
 8   v_elapsedtime     33193 non-null  float64       
 9   v_totalproblem    33193 non-null  object        
 10  v_currentproblem  33193 non-null  object        
 11  v_missed          33193 non-null  object        
 12  v_solved          33193 non-null  object        
 13  v_monsterid       33193 non-null  object        
 14  v_dailyunitindex  3319

# Load the data file into sqlite3

In [11]:
conn = sqlite3.connect(":memory:")

In [25]:
# make the monster id sorting friendly
quiz_df['monster'] = quiz_df['v_monsterid'].apply(lambda input: '.'.join([ele.zfill(2) if ele.isnumeric() else ele for ele in input.split('.')]))

# has the session been aborted?
quiz_df['abort'] = (quiz_df['v_totalproblem'] > quiz_df['v_currentproblem'])


In [ ]:
# sort the quiz_df and export the 'raw' data
quiz_df = quiz_df.sort_values(by=['accountid', 'userid', 'regdate'], ignore_index=True)

quiz_df.to_csv('/content/gdrive/MyDrive/ongoing/todo_math/monster_quiz_session.csv', index=False)
files.download('/content/gdrive/MyDrive/ongoing/todo_math/monster_quiz_session.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
%time quiz_df.to_sql('quiz', conn, index=False, if_exists='replace')

CPU times: user 1.33 s, sys: 11.7 ms, total: 1.34 s
Wall time: 1.35 s


# Generate the numbers for monster quiz usage

In [27]:
query = """
  WITH total_data AS
  (
    SELECT
      monster
      , COUNT(1) AS total_play_cnt
      , CAST(SUM(v_elapsedtime) AS INT) / 60 AS total_play_time_min
      , COUNT(DISTINCT userid) AS total_user_cnt
    FROM quiz
    WHERE v_monsterid IS NOT NULL
    GROUP BY 1
  ), 

  finish_data AS
  (
    SELECT
      monster
      , COUNT(1) AS finish_play_cnt
      , CAST(SUM(v_elapsedtime) AS INT) / 60 AS finish_play_time_min
      , CAST(AVG(v_elapsedtime) AS INT) AS finish_time_avg_sec
      , COUNT(DISTINCT userid) AS finish_user_cnt
      , AVG(CAST(v_solved AS FLOAT) / v_totalproblem) AS finish_avg_acc
    FROM quiz
    WHERE v_monsterid IS NOT NULL
      AND abort = FALSE
    GROUP BY 1
  ),

  abort_data AS
  (
    SELECT
      monster
      , COUNT(1) AS abort_play_cnt
      , COUNT(DISTINCT userid) AS abort_user_cnt
    FROM quiz
    WHERE v_monsterid IS NOT NULL
      AND abort = TRUE
    GROUP BY 1
  ),

  count_user AS
  (
    SELECT
      monster
      , SUM(CASE WHEN finish_cnt > 0 AND abort_cnt = 0 THEN 1 ELSE 0 END) AS finish_only_user_cnt
      , SUM(CASE WHEN finish_cnt > 0 AND abort_cnt > 0 THEN 1 ELSE 0 END) AS abort_and_finish_user_cnt
      , SUM(CASE WHEN finish_cnt = 0 AND abort_cnt > 0 THEN 1 ELSE 0 END) AS abort_only_user_cnt    
    FROM (
      SELECT 
        monster
        , userid
        , SUM(CASE WHEN abort = FALSE THEN 1 ELSE 0 END) AS finish_cnt
        , SUM(CASE WHEN abort = TRUE THEN 1 ELSE 0 END) AS abort_cnt
      FROM quiz
      GROUP BY 1, 2
    )
    GROUP BY 1
  )  

  SELECT
    t.*
    , IFNULL(f.finish_play_cnt, 0) AS finish_play_cnt   
    , IFNULL(f.finish_play_time_min, 0) AS finish_play_time_min   
    , IFNULL(f.finish_time_avg_sec, 0) AS finish_time_avg_sec   
    , IFNULL(f.finish_avg_acc, 0) AS finish_avg_acc       
    , IFNULL(f.finish_user_cnt, 0) AS finish_user_cnt   
    , IFNULL(a.abort_play_cnt, 0) AS abort_play_cnt   
    , IFNULL(a.abort_user_cnt, 0) AS abort_user_cnt   
    , CAST(a.abort_play_cnt AS FLOAT) / t.total_play_cnt AS abort_play_rate
    , CAST(a.abort_user_cnt AS FLOAT) / t.total_user_cnt AS abort_user_rate
    , c.finish_only_user_cnt 
    , CAST(c.finish_only_user_cnt AS FLOAT) / t.total_user_cnt AS finish_only_user_rate
    , c.abort_and_finish_user_cnt 
    , CAST(c.abort_and_finish_user_cnt AS FLOAT) / t.total_user_cnt AS abort_and_finish_user_rate
    , c.abort_only_user_cnt 
    , CAST(c.abort_only_user_cnt AS FLOAT) / t.total_user_cnt AS abort_only_user_rate

  FROM total_data t
    LEFT JOIN finish_data f ON t.monster = f.monster
    LEFT JOIN abort_data a ON t.monster = a.monster
    LEFT JOIN count_user c ON t.monster = c.monster

"""

%time quiz_abort_df = pd.read_sql(query, conn)
quiz_abort_df

CPU times: user 748 ms, sys: 4.31 ms, total: 752 ms
Wall time: 753 ms


,monster,total_play_cnt,total_play_time_min,total_user_cnt,finish_play_cnt,finish_play_time_min,finish_time_avg_sec,finish_avg_acc,finish_user_cnt,abort_play_cnt,abort_user_cnt,abort_play_rate,abort_user_rate,finish_only_user_cnt,finish_only_user_rate,abort_and_finish_user_cnt,abort_and_finish_user_rate,abort_only_user_cnt,abort_only_user_rate
0,A.M.01,7968,4377,6846,7297,4167,34,0.777214,6773,671,405,0.084212,0.059159,6441,0.940841,332,0.048495,73,0.010663
1,A.M.02,4106,2402,3384,3727,2283,36,0.668706,3353,379,249,0.092304,0.073582,3135,0.926418,218,0.064421,31,0.009161
2,A.M.03,3976,4259,3189,3358,3717,66,0.786480,3087,618,323,0.155433,0.101286,2866,0.898714,221,0.069301,102,0.031985
3,A.M.04,2993,1839,2624,2781,1774,38,0.792017,2607,212,144,0.070832,0.054878,2480,0.945122,127,0.048399,17,0.006479
4,A.M.05,3418,3180,2256,2343,2700,69,0.664533,2225,1075,436,0.314511,0.193262,1820,0.806738,405,0.179521,31,0.013741
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,F.M.07,724,846,547,618,776,75,0.925770,545,106,61,0.146409,0.111517,486,0.888483,59,0.107861,2,0.003656
57,F.M.08,575,278,474,527,267,30,0.972676,474,48,31,0.083478,0.065401,443,0.934599,31,0.065401,0,0.000000
58,F.M.09,710,1037,455,553,904,98,0.893309,453,157,83,0.221127,0.182418,372,0.817582,81,0.178022,2,0.004396
59,F.M.10,1059,1457,614,790,1274,96,0.911027,609,269,108,0.254013,0.175896,506,0.824104,103,0.167752,5,0.008143


In [28]:
quiz_abort_df.to_csv('/content/gdrive/MyDrive/ongoing/todo_math/TM_monster_quiz.csv', index=False)
files.download('/content/gdrive/MyDrive/ongoing/todo_math/TM_monster_quiz.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# First try, Second try?, aborts?

In this case, we need the full history of monster quiz from the start, so we  need to look at the data from only the users who started after 04-07, when we started to collect the monster data.




In [29]:
query = f"""
  SELECT 
    installid
    , accountid
    , userid
    , platform
    , version
    , regdate
    , event
    , v_gamename
    , v_elapsedtime
    , v_totalproblem
    , v_currentproblem
    , v_missed
    , v_solved
    , v_monsterid
    , v_dailyunitindex
  FROM dw.tb_fact_todomath_log
  WHERE server_yyyymmdd > '2021-03'
    AND v_gamename = 'MonsterQuiz'
    AND event = 'tmGameComplete'
    AND v_mode = '4'
    AND v_monsterid IS NOT NULL
    AND todomath_regdate > DATE '2021-04-07'
  ORDER BY 2, 3, 6
"""

%time quiz_after_0407 = pd.read_sql(query, athena)
quiz_after_0407

CPU times: user 3.49 s, sys: 88.1 ms, total: 3.57 s
Wall time: 28.9 s


,installid,accountid,userid,platform,version,regdate,event,v_gamename,v_elapsedtime,v_totalproblem,v_currentproblem,v_missed,v_solved,v_monsterid,v_dailyunitindex
0,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-17 10:57:27,tmGameComplete,MonsterQuiz,16.877234,5,0,5,0,T.M.0,1-0
1,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-17 10:59:20,tmGameComplete,MonsterQuiz,107.633120,5,5,0,5,T.M.0,1-0
2,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-18 08:11:30,tmGameComplete,MonsterQuiz,27.847017,5,5,2,3,A.M.1,1-1
3,686447BA-5D2A-421C-AB5D-FFB4D98DF3F7,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,iOS,5.21.8,2021-04-21 09:23:16,tmGameComplete,MonsterQuiz,29.845467,5,5,2,3,A.M.2,1-2
4,A956C9A7-D3F3-4040-9E3E-C254417A3547,00044e10-a687-11eb-ade4-df0aaaf64726,BB2D563C-AE77-4A30-AC7D-63812F1D763D,iOS,5.21.8,2021-04-26 12:14:26,tmGameComplete,MonsterQuiz,38.327930,5,5,1,4,T.M.0,2-0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33188,A7685A30-3F3D-4F49-9150-EAF60A2665A8,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,iOS,5.21.8,2021-04-24 03:37:15,tmGameComplete,MonsterQuiz,40.797104,5,5,0,5,A.M.4,1-4
33189,A7685A30-3F3D-4F49-9150-EAF60A2665A8,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,iOS,5.21.8,2021-04-24 03:56:57,tmGameComplete,MonsterQuiz,59.105620,5,5,1,4,A.M.5,1-5
33190,A7685A30-3F3D-4F49-9150-EAF60A2665A8,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,iOS,5.21.8,2021-04-24 05:34:10,tmGameComplete,MonsterQuiz,67.862045,5,5,3,2,A.M.6,1-6
33191,A7685A30-3F3D-4F49-9150-EAF60A2665A8,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,iOS,5.21.8,2021-04-24 06:14:35,tmGameComplete,MonsterQuiz,37.162160,5,5,2,3,A.M.7,1-7


In [44]:
# make the monster id sorting friendly
quiz_after_0407['monster'] = quiz_after_0407['v_monsterid'].apply(lambda input: '.'.join([ele.zfill(2) if ele.isnumeric() else ele for ele in input.split('.')]))

In [45]:
%time quiz_after_0407.to_sql('quiz', conn, index=False, if_exists='replace')

CPU times: user 328 ms, sys: 1.75 ms, total: 330 ms
Wall time: 334 ms


In [55]:
query = """
  WITH user AS
  (
    SELECT
      accountid
      , userid
      , monster
      , COUNT(1) AS try_cnt
      , SUM(CASE WHEN v_totalproblem = v_currentproblem THEN 1 ELSE 0 END) AS finish_cnt
      , SUM(CASE WHEN (v_totalproblem = v_currentproblem) AND (v_solved >= '4') THEN 1 ELSE 0 END) AS pass_cnt
    FROM quiz
    GROUP BY 1, 2, 3
    HAVING monster < 'T'
    ORDER BY 1, 2, 3
  )

  SELECT * FROM user

"""


"""
  SELECT
    monster
    , COUNT(DISTINCT userid) AS try_user

    , SUM(CASE WHEN pass_cnt > 0 THEN 1 ELSE 0 END) AS pass_user
    , 100*CAST(SUM(CASE WHEN pass_cnt > 0 THEN 1 ELSE 0 END) AS REAL)
      / COUNT(DISTINCT userid) AS pass_user_pcnt

    , SUM(CASE WHEN (try_cnt = 1) AND (finish_cnt = 1) THEN 1 ELSE 0 END) AS one_try_user
    , SUM(CASE WHEN (try_cnt = 1) AND (finish_cnt = 1) AND (pass_cnt = 1) THEN 1 ELSE 0 END) AS one_try_pass_user
    , 100*CAST(SUM(CASE WHEN (try_cnt = 1) AND (finish_cnt = 1) AND (pass_cnt = 1) THEN 1 ELSE 0 END) AS REAL)
      / SUM(CASE WHEN (try_cnt = 1) AND (finish_cnt = 1) THEN 1 ELSE 0 END) AS one_try_pass_pcnt

    , SUM(CASE WHEN (try_cnt = 2) AND (finish_cnt = 2) THEN 1 ELSE 0 END) AS two_try_user
    , SUM(CASE WHEN (try_cnt = 2) AND (finish_cnt = 2) AND (pass_cnt = 1) THEN 1 ELSE 0 END) AS two_try_pass_user
    , 100*CAST(SUM(CASE WHEN (try_cnt = 2) AND (finish_cnt = 2) AND (pass_cnt >= 1) THEN 1 ELSE 0 END) AS REAL)
      / SUM(CASE WHEN (try_cnt = 2) AND (finish_cnt = 2) THEN 1 ELSE 0 END) AS two_try_pass_pcnt

    , SUM(CASE WHEN try_cnt > finish_cnt THEN 1 ELSE 0 END) AS abort_user
    , 100*CAST(SUM(CASE WHEN try_cnt > finish_cnt THEN 1 ELSE 0 END) AS REAL)
      / COUNT(DISTINCT userid) AS abort_user_pcnt
  FROM user
  GROUP BY 1
  ORDER BY 1
"""

%time pass_df = pd.read_sql(query, conn)
pass_df

CPU times: user 98.7 ms, sys: 755 µs, total: 99.4 ms
Wall time: 106 ms


,accountid,userid,monster,try_cnt,finish_cnt,pass_cnt
0,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,A.M.01,1,1,0
1,0001c2f0-9f69-11eb-ac9f-e938389cbd2c,82BF4B13-48D8-4CD0-937F-CE0E492EAA84,A.M.02,1,1,0
2,00065260-9d9e-11eb-8846-0d7e0b3f0ed6,495BED43-AE18-4D15-82AC-9EFD3C0832B8,B.M.01,1,1,1
3,00065260-9d9e-11eb-8846-0d7e0b3f0ed6,495BED43-AE18-4D15-82AC-9EFD3C0832B8,B.M.02,1,1,1
4,00065260-9d9e-11eb-8846-0d7e0b3f0ed6,495BED43-AE18-4D15-82AC-9EFD3C0832B8,B.M.03,1,1,1
...,...,...,...,...,...,...
19445,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,A.M.03,1,1,1
19446,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,A.M.04,1,1,1
19447,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,A.M.05,1,1,1
19448,kakao_ff63e690-567d-11eb-94d9-c7aa68f2c331,44C5E9C8-2924-45E7-B7CB-3DCD5067CE12,A.M.06,1,1,0


In [47]:
pass_df.to_csv('/content/gdrive/MyDrive/ongoing/todo_math/TM_monster_pass.csv', index=False)
files.download('/content/gdrive/MyDrive/ongoing/todo_math/TM_monster_pass.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
pass_df.to_csv('/content/gdrive/MyDrive/ongoing/todo_math/TM_monster_user_agg.csv', index=False)
files.download('/content/gdrive/MyDrive/ongoing/todo_math/TM_monster_user_agg.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
tmp_df['latest_yyyymmdd'].value_counts()

2021-02-02    36659
2021-02-01    19940
2021-01-31    16251
2021-01-30    10479
2021-01-29    10053
2021-01-28     8711
2021-01-27     7741
2021-01-24     7489
2021-01-26     6825
2021-01-25     6353
Name: latest_yyyymmdd, dtype: int64

# [더이상 사용하지 않음] userinfo에서 몬스터 데이터 추출

In [ ]:
# get the userinfo

query = f"""
  WITH 
  latest_info AS
    (
      SELECT
        accountid, 
        userid,
        max(timestamp) as latest,
        max(server_yyyymmdd) AS latest_yyyymmdd
      FROM dw.tb_fact_todomath_log
      WHERE server_yyyymmdd > '{weekago}'
        AND userid is not NULL
        AND accountid is not NULL
      GROUP BY accountid, userid
    ),

  daily_log AS
    (
      SELECT
        accountid,
        userid,
        timestamp,
        deviceinfo_localecountry AS country,
        platform,
        payment,
        userinfo.reference_str,
        userinfo_regdate
      FROM dw.tb_fact_todomath_log
      WHERE server_yyyymmdd > '{weekago}'
    )
  
  SELECT
    li.accountid,
    li.userid,
    dl.timestamp,
    dl.country,
    dl.platform,
    dl.payment,
    dl.reference_str,
    dl.userinfo_regdate
  FROM latest_info li
  LEFT JOIN daily_log dl 
    ON li.accountid = dl.accountid AND li.userid = dl.userid AND li.latest = dl.timestamp;
"""

#%time tmp_df = pd.read_sql(query, athena)

if True:
  %time user_df = pd.read_sql(query, athena)
  user_df.info()
  user_df.to_csv('userinfo_' + yesterday + '.csv.gz', index=False)
  files.download('userinfo_' + yesterday + '.csv.gz')


CPU times: user 27.3 s, sys: 2.11 s, total: 29.4 s
Wall time: 3min 8s
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105319 entries, 0 to 105318
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   accountid         105319 non-null  object        
 1   userid            105319 non-null  object        
 2   timestamp         105319 non-null  object        
 3   country           105318 non-null  object        
 4   platform          105319 non-null  object        
 5   payment           105319 non-null  int64         
 6   reference_str     103452 non-null  object        
 7   userinfo_regdate  103452 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 6.4+ MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
user_df

,accountid,userid,timestamp,country,platform,payment,reference_str,userinfo_regdate
0,70b815a0-32d4-11eb-a838-ff7380eb9ddf,2370d06d-e535-4853-a539-8c4ef1c14bf4,1612264811,CN,android,1,"{""accountId"":""70b815a0-32d4-11eb-a838-ff7380eb...",2020-12-24 10:31:21
1,google_106100351681354616157,e59384e2-e328-4ca5-8271-33d3fa1ab2ed,1611968071,KR,android,0,"{""accountId"":""google_106100351681354616157"",""b...",2021-01-17 13:43:53
2,6f717790-b9f6-11e7-a772-5f07b3c6a753,41AEB86F-24D9-4043-896D-2B5F825565CE,1611968345,CN,iOS,0,"{""accountId"":""6f717790-b9f6-11e7-a772-5f07b3c6...",2021-01-27 06:36:42
3,8d661350-5b15-11eb-b97e-2573a6df6aad,C80632C8-81D5-4A62-875B-DAB8ECD021C1,1611970729,US,iOS,1,"{""accountId"":""8d661350-5b15-11eb-b97e-2573a6df...",2021-01-20 11:47:43
4,d99ece50-610b-11eb-8269-01c1caeeae77,9877bd6f-ddfd-4b2f-838f-27f8741f93d1,1612274059,CN,android,1,"{""accountId"":""d99ece50-610b-11eb-8269-01c1caee...",2021-01-28 01:54:01
...,...,...,...,...,...,...,...,...
105314,c979ba30-c32e-11e8-93c7-5f90c202e106,c752c2dc-9261-48cb-aef3-2ea41914f1fd,1612299033,US,android,0,"{""accountId"":"""",""birthMonth"":1,""birthYear"":200...",2021-02-02 20:48:04
105315,c979ba30-c32e-11e8-93c7-5f90c202e106,6e613293-ff4e-4c61-bcf3-afcad02cfbc8,1612298473,US,android,0,"{""accountId"":"""",""birthMonth"":1,""birthYear"":200...",2021-02-02 20:35:41
105316,apple_001184.a78ef8e4c0da47cdbe5e01057bd54a36....,3E5B5BA3-54D9-4640-A265-BFB46444ABE0,1612280779,CN,iOS,0,"{""accountId"":""apple_001184.a78ef8e4c0da47cdbe5...",2020-11-01 12:16:45
105317,9e85a850-e6b5-11ea-8193-ada8190a9063,12d1a2ac-c94d-42b8-90e4-87d8ae357dfd,1612232005,CN,android,0,"{""accountId"":""9e85a850-e6b5-11ea-8193-ada8190a...",2019-08-17 13:55:06


## 사용자 단위 데이터

In [ ]:
user_df = pd.read_csv('userinfo_' + yesterday + '.csv.gz')

In [ ]:
# Some info are encoded in the reference_str
# so this function loads the json string and extracts the info

def extract_info(key, json_str):
  if isinstance(json_str, str):
    tmp_data = json.loads(json_str)
    
    if key == 'level':
      try:
        return tmp_data['dailyInfo']['dailyLevel']
      except:
        return -1
    
    elif key == 'star':
      # some users may have multiple devices
      # with different number of stars
      try:
        return sum(tmp_data['numStars'].values())
      except:
        return 0
    
    elif key == 'age_month':
      try:
        diff = relativedelta.relativedelta(datetime.now(), datetime(tmp_data['birthYear'], tmp_data['birthMonth'], 1))
        if diff.years > 15:
          return np.nan 
        else:
          return diff.years*12 + diff.months
      except:
        return np.nan

    elif key == 'lasttime':
      try:
        return tmp_data['lastPlayTimestamp']['timestamp']
      except:
        return ''

    elif key == 'install':
      try:
        return tmp_data['lastPlayTimestamp']['installId']
      except:
        return ''

    elif key == 'playtime':
      try:
        return round(sum(list(tmp_data['playingTime'].values())) / 60)
      except:
        return np.nan

    elif key == 'check1':
      return len(tmp_data['numStars'])
    
    else:
      return np.nan
  else:
    return np.nan

In [ ]:
extract_info('age_month', user_df['reference_str'][0])

87

In [ ]:
%%time 
from functools import partial 

user_df['star'] = user_df['reference_str'].map(partial(extract_info, 'star'))
user_df['level'] = user_df['reference_str'].map(partial(extract_info, 'level'))
#user_df['installid'] = user_df['reference_str'].map(partial(extract_info, 'install'))
user_df['playtime_min'] = user_df['reference_str'].map(partial(extract_info, 'playtime'))
user_df['age_month'] = user_df['reference_str'].map(partial(extract_info, 'age_month'))

CPU times: user 39.4 s, sys: 25.7 ms, total: 39.4 s
Wall time: 39.5 s


In [ ]:
#user_df.drop(user_df.columns[0], axis=1, inplace=True)
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105319 entries, 0 to 105318
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   accountid         105319 non-null  object 
 1   userid            105319 non-null  object 
 2   timestamp         105319 non-null  int64  
 3   country           105294 non-null  object 
 4   platform          105319 non-null  object 
 5   payment           105319 non-null  int64  
 6   reference_str     103452 non-null  object 
 7   userinfo_regdate  103452 non-null  object 
 8   star              103452 non-null  float64
 9   level             103452 non-null  float64
 10  playtime_min      103438 non-null  float64
 11  age_month         80195 non-null   float64
dtypes: float64(4), int64(2), object(6)
memory usage: 9.6+ MB


In [ ]:
#valid_idx = (user_df['star'] < 30000)
#user_df[valid_idx]['star'].hist(bins=100)

## userinfo 에서 몬스터 데이터 추출

In [ ]:
%%time

monster_log = []

for row in zip(user_df['accountid'], user_df['userid'], user_df['reference_str']):
  try:
    tmp_data = json.loads(row[2])
    if 'monsterInfo' in tmp_data.keys():
      for key in tmp_data['monsterInfo'].keys():
        tmp_mon = tmp_data['monsterInfo'][key]
        # format: accountid, userid, monsterid, last_timestamp, last_score, num_finish, num_fail, max_score, is_captured, captured_date
        num_finish = sum(tmp_mon['numFail'].values()) + sum(tmp_mon['numSuccess'].values())
        num_fail = sum(tmp_mon['numFail'].values())
        monster_log.append([row[0], row[1], key, pd.to_datetime(tmp_mon['lastTimestamp'], format='%Y-%m-%d %H:%M:%S'), 
                            tmp_mon['lastScore'], num_finish, num_fail, tmp_mon['maxScore'], tmp_mon['isCaptured'], 
                            pd.to_datetime(tmp_mon['dateCaptured'], format='%Y-%m-%d %H:%M:%S')])
  except:
    pass        



In [ ]:
#len(monster_log)
monster_df = pd.DataFrame(monster_log, 
                          columns=['accountid', 'userid', 'monsterid', 'last_timestamp', 'last_score', 'num_finish', 'num_fail', 'max_score', 'is_captured', 'captured_date'])

## monster 정보에 user 정보 결합


In [ ]:
monster_df = monster_df.merge(user_df.drop(columns='reference_str', axis=1), on=['accountid', 'userid'])

In [ ]:
monster_df.info()
monster_df.to_csv('monster_' + yesterday + '.csv', index=False)
files.download('monster_' + yesterday + '.csv')

<class 'pandas.core.frame.DataFrame'>
Int64Index: 961851 entries, 0 to 961850
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   accountid         961851 non-null  object        
 1   userid            961851 non-null  object        
 2   monsterid         961851 non-null  object        
 3   last_timestamp    961851 non-null  datetime64[ns]
 4   last_score        961851 non-null  int64         
 5   num_finish        961851 non-null  int64         
 6   num_fail          961851 non-null  int64         
 7   max_score         961851 non-null  int64         
 8   is_captured       961851 non-null  bool          
 9   captured_date     961851 non-null  datetime64[ns]
 10  timestamp         961851 non-null  int64         
 11  country           961100 non-null  object        
 12  platform          961851 non-null  object        
 13  payment           961851 non-null  int64         
 14  user

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 몬스터 별 집계

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")

In [ ]:
monster_df.to_sql('monster', conn, index=False, if_exists='replace')

In [ ]:
query = """
  SELECT 
    monsterid,
    SUM(num_finish) AS total_finish,
    SUM(num_fail) AS total_fail,
    CAST(SUM(num_fail) AS FLOAT) / SUM(num_finish) AS fail_prob,
    AVG(is_captured) AS capt_prob
  FROM monster
  GROUP BY monsterid;
"""

results = pd.read_sql(query, conn)
results

,monsterid,total_finish,total_fail,fail_prob,capt_prob
0,A.M.1,71649,24580,0.343061,0.860834
1,A.M.10,22655,8517,0.375944,0.831310
2,A.M.2,70758,34293,0.484652,0.787200
3,A.M.3,50656,14080,0.277953,0.900548
4,A.M.4,42438,14129,0.332933,0.867587
...,...,...,...,...,...
138,SC.KPP.6,2701,1185,0.438726,0.844286
139,SC.KPP.7,1893,229,0.120972,0.972154
140,SC.KPP.8,1641,590,0.359537,0.848000
141,SC.KPP.9,1311,246,0.187643,0.920935


In [ ]:
results.to_csv('group_by_monster_' + yesterday + '.csv', index=False)
files.download('group_by_monster_' + yesterday + '.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>